# Automated Marine Debris Detection using Deep Learning

## 1. Introduction and Problem Statement

## 2. Data Sources

## 3. Exploratory Data Analysis (EDA)

### 3.1 Dataset Overview

### 3.2 Class Distribution

### 3.3 Image and Annotation Visualisation

### 3.4 Preprocessing and Data Splitting

Reuse the committed video-level train/test split for comparable experiments. The audit checks train/test separation and reports overlap with the supplied validation set. Regenerating a split invalidates checkpoints trained on the previous split.

In [ ]:
import json
import random
from pathlib import Path
from collections import defaultdict
from help_funciton.evaluation_protocol import audit_splits

OUTPUT_DIR = Path("instance_version")
SPLIT_SEED = 42
REGENERATE_SPLIT = False  # Changing this requires retraining every model.
split_paths = [OUTPUT_DIR / "instances_train.json", OUTPUT_DIR / "instances_test.json"]
exists = [p.exists() for p in split_paths]
if any(exists) and not all(exists) and not REGENERATE_SPLIT:
    raise RuntimeError("Only one split file exists. Restore both committed split files.")

if REGENERATE_SPLIT or not all(exists):
    with (OUTPUT_DIR / "instances_train_trashcan.json").open(encoding="utf-8") as f:
        data = json.load(f)
    video_groups = defaultdict(list)
    image_categories = defaultdict(set)
    for image in data["images"]:
        video_groups[image["file_name"].split("_frame")[0]].append(image)
    for ann in data["annotations"]:
        image_categories[ann["image_id"]].add(ann["category_id"])
    all_categories = {c["id"] for c in data["categories"]}
    rng = random.Random(SPLIT_SEED)
    for attempt in range(200):
        video_ids = sorted(video_groups)
        rng.shuffle(video_ids)
        selected, count = set(), 0
        for video_id in video_ids:
            if count >= 1000:
                break
            selected.add(video_id)
            count += len(video_groups[video_id])
        test_images = [im for video, group in video_groups.items() if video in selected for im in group]
        train_images = [im for video, group in video_groups.items() if video not in selected for im in group]
        train_cats = set().union(*(image_categories[im["id"]] for im in train_images))
        test_cats = set().union(*(image_categories[im["id"]] for im in test_images))
        if train_cats == test_cats == all_categories:
            break
    else:
        raise RuntimeError("Failed to find a video-disjoint split with all categories.")
    for output_path, images in zip(split_paths, [train_images, test_images]):
        image_ids = {im["id"] for im in images}
        split = {"info": data.get("info", {}), "licenses": data.get("licenses", []),
                 "images": images, "categories": data["categories"],
                 "annotations": [a for a in data["annotations"] if a["image_id"] in image_ids]}
        output_path.write_text(json.dumps(split), encoding="utf-8")
else:
    print("Reusing the committed train/test split; no annotations were overwritten.")

split_data = [json.loads(p.read_text(encoding="utf-8")) for p in split_paths]
validation_data = json.loads((OUTPUT_DIR / "instances_val_trashcan.json").read_text(encoding="utf-8"))
split_audit = audit_splits(split_data[0], split_data[1], validation_data)
print(json.dumps(split_audit, indent=2))

## 4. Models and Methods

### 4.1 Faster R-CNN

### 4.2 Mask R-CNN

In [ ]:
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn_v2,
    MaskRCNN_ResNet50_FPN_V2_Weights
)
from torchvision.models.detection.faster_rcnn import (
    FastRCNNPredictor
)
from torchvision.models.detection.mask_rcnn import (
    MaskRCNNPredictor
)

def create_model(num_classes, pretrained=True):
    model = maskrcnn_resnet50_fpn_v2(
        weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT if pretrained else None,
        weights_backbone=None
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features

    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # now get the number of input features for the mask classifier
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    # and replace the mask predictor with a new one
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask,
        hidden_layer,
        num_classes
    )
    return model

Baseline

In [ ]:
def load_baseline_model(device):
    model = maskrcnn_resnet50_fpn_v2(
            weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
        )
    model.to(device=device)
    model.eval()

    return model

## 5. Experimental Setup

### 5.1 Training Configuration

In [ ]:
import torch
from pycocotools.coco import COCO
from help_funciton.marine_dataset import MarineDataset

Transformer

In [ ]:
from torchvision.transforms import v2 as T
def get_transform(train):
    transforms = []
    if train:
        transforms.append(T.RandomHorizontalFlip(0.5))
    transforms.append(T.ToDtype(torch.float, scale=True))
    transforms.append(T.ToPureTensor())
    return T.Compose(transforms)

Dataset setup

In [ ]:
from help_funciton.engine import train_one_epoch, evaluate

# train on the accelerator or on the CPU, if an accelerator is not available
device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")

# One shared foreground label mapping for train, validation and test.
image_dir = "instance_version/train"
annotation_file = "instance_version/instances_train.json"
category_id = COCO(annotation_file).getCatIds()
num_classes = len(category_id) + 1
dataset = MarineDataset(image_dir, annotation_file, category_id, transforms=get_transform(train=False))
image_dir_val = "instance_version/val"
annotation_file_val = "instance_version/instances_val_trashcan.json"
category_id_val = category_id
num_classes_val = len(category_id_val) + 1
dataset_val = MarineDataset(image_dir_val, annotation_file_val, category_id_val, transforms=get_transform(train=False))

dataset_trans = MarineDataset(image_dir, annotation_file, category_id, transforms=get_transform(train=True))


# define training and validation data loaders
def collate_fn(batch):
    return tuple(zip(*batch))
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

data_loader_val = torch.utils.data.DataLoader(
    dataset_val,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn
)

data_loader_trans = torch.utils.data.DataLoader(
    dataset_trans,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

Train sgd+steplr

In [ ]:
model = create_model(num_classes)

# move model to the right device
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    weight_decay=0.0005,
    momentum=0.9
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

# let's train it just for 2 epochs
num_epochs = 15

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # Monitor validation only; reserve test metrics for final reporting.
    evaluate(model, data_loader_val, device=device)

Train adamw+cosineAnnealingLR

In [ ]:
model_adamw = create_model(num_classes)

model_adamw.to(device)

# construct an optimizer
params = [p for p in model_adamw.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    params,
     lr=0.0001,
    weight_decay=0.0001
)

num_epochs = 15

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=1e-6
)

# let's train it just for 2 epochs


for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model_adamw, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # Monitor validation only; reserve test metrics for final reporting.
    evaluate(model_adamw, data_loader_val, device=device)

train sgd+stepLR using trans data

In [ ]:
# get the model using our helper function
model_sgd_trans = create_model(num_classes)

# move model to the right device
model_sgd_trans.to(device)

# construct an optimizer
params = [p for p in model_sgd_trans.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    weight_decay=0.0005,
    momentum=0.9
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

# let's train it just for 2 epochs
num_epochs = 15

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model_sgd_trans, optimizer, data_loader_trans, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # Monitor validation only; reserve test metrics for final reporting.
    evaluate(model_sgd_trans, data_loader_val, device=device)

Test dataset setup

In [ ]:
image_dir_test = "instance_version/train"
annotation_file_test = "instance_version/instances_test.json"
category_id_test = category_id
num_classes_test = len(category_id_test) + 1
dataset_test = MarineDataset(image_dir_test, annotation_file_test, category_id_test, transforms=get_transform(train=False))


data_loader_test = torch.utils.data.DataLoader(
    dataset_test,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn
)

### 5.2 Evaluation Metrics

Report bounding-box (`bbox`) and instance-mask (`segm`) metrics separately:
- **mAP** averages precision over IoU 0.50:0.05:0.95; **AP50/AP75** use one IoU threshold.
- **AR1/AR10/AR100** average recall at the COCO detection limits. Small/medium/large metrics use the original annotation areas.
- Per-class AP and precision-recall curves use the same training-label mapping. Undefined metrics (no ground truth) appear as NaN, not zero. Scores are on a 0-1 scale.

All predictions returned by the model are passed to COCO evaluation, without an extra visualization threshold. Mask probabilities are binarized at 0.5; the model's built-in score/NMS settings still apply. All compared models must use the same inference settings and held-out split. Choose hyperparameters/checkpoints using validation only; do not choose a winner from test scores.

The unmodified COCO-pretrained model from `load_baseline_model` predicts COCO categories, not TrashCAN categories, and is intentionally excluded from the numerical comparison. It can be inspected qualitatively using its own COCO class names. Faster R-CNN is still unimplemented in section 4.1 and has no results to report.

References: [COCO evaluator](https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py), [Torchvision Mask R-CNN weights](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.maskrcnn_resnet50_fpn_v2.html).

In [ ]:
from IPython.display import display
import hashlib
import torchvision
import pandas as pd
import matplotlib.pyplot as plt
from help_funciton.evaluation import (
    summarize_evaluators, per_class_metrics, plot_precision_recall,
    plot_predictions, plot_model_comparison, export_evaluation_results,
)

# Optional: supply locally trained state_dict files to evaluate after restarting
# the kernel. Skip training cells when using these checkpoints. Each checkpoint
# must use this exact committed split and the category_id training-label order.
MODEL_CHECKPOINTS = {}  # e.g. {"Mask R-CNN / SGD": "checkpoints/sgd.pth"}
model_variables = {
    "Mask R-CNN / SGD": "model",
    "Mask R-CNN / AdamW": "model_adamw",
    "Mask R-CNN / SGD + flip": "model_sgd_trans",
}
if set(MODEL_CHECKPOINTS) - set(model_variables):
    raise ValueError("Unknown checkpoint name; use a key from model_variables.")
models_to_evaluate = {}
for name, variable in model_variables.items():
    if name in MODEL_CHECKPOINTS:
        candidate = create_model(num_classes, pretrained=False)
        checkpoint = torch.load(MODEL_CHECKPOINTS[name], map_location="cpu", weights_only=True)
        candidate.load_state_dict(checkpoint.get("model_state_dict", checkpoint))
    else:
        candidate = globals().get(variable)
    if candidate is not None:
        models_to_evaluate[name] = candidate.to(device)
if not models_to_evaluate:
    raise RuntimeError("Train at least one model in section 5.1 or provide MODEL_CHECKPOINTS.")

class_names = {dataset.cat_to_label[c["id"]]: c["name"] for c in dataset.coco.dataset["categories"]}
assert dataset.cat_to_label == dataset_val.cat_to_label == dataset_test.cat_to_label
evaluators = {}
for name, candidate in models_to_evaluate.items():
    print(f"\nEvaluating {name} on {len(dataset_test)} held-out images")
    evaluators[name] = evaluate(candidate, data_loader_test, device=device)
summary = summarize_evaluators(evaluators)
display(summary.round(4))

## 6. Results

### 6.1 Quantitative Results

Tables are computed from the completed evaluations above. No placeholder or precomputed model scores are included. Blank CSV cells indicate undefined metrics.

In [ ]:
OUTPUT_DIR_EVAL = Path("evaluation_results") / "held_out_test"
report = export_evaluation_results(evaluators, OUTPUT_DIR_EVAL, class_names)
display(report["summary"].round(4))
display(report["per_class"].round(4))

# Save enough context to identify the annotations and label order used here.
annotation_paths = [annotation_file, annotation_file_val, annotation_file_test]
metadata = {
    "split": "held_out_test", "split_audit": split_audit,
    "category_ids_in_training_order": list(category_id),
    "class_names": class_names, "models": list(models_to_evaluate),
    "checkpoints": {name: str(value) for name, value in MODEL_CHECKPOINTS.items()},
    "checkpoint_sha256": {name: hashlib.sha256(Path(value).read_bytes()).hexdigest()
                          for name, value in MODEL_CHECKPOINTS.items()},
    "torch_version": torch.__version__, "torchvision_version": torchvision.__version__,
    "annotation_sha256": {str(p): hashlib.sha256(Path(p).read_bytes()).hexdigest() for p in annotation_paths},
    "inference_settings": {name: {key: getattr(candidate.roi_heads, key) for key in
                           ("score_thresh", "nms_thresh", "detections_per_img")}
                           for name, candidate in models_to_evaluate.items()},
    "extra_metric_score_threshold": None, "mask_threshold": 0.5,
}
(OUTPUT_DIR_EVAL / "evaluation_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(f"Saved tables, PR curves, comparison plot and evaluation metadata to {OUTPUT_DIR_EVAL.resolve()}")

### 6.2 Qualitative Results

Ground truth and predicted boxes/masks are shown side by side. Inspect missed small objects, duplicate detections, class confusion and mask boundaries; the fixed sample is illustrative, not a substitute for full-set metrics.

In [ ]:
import numpy as np

# Use the same reproducible sample for every model. This threshold is for
# display only and never changes the metrics or exported COCO PR curves.
sample_indices = np.random.default_rng(42).choice(len(dataset_test), min(3, len(dataset_test)), replace=False).tolist()
for index, (name, candidate) in enumerate(models_to_evaluate.items(), start=1):
    fig = plot_predictions(candidate, dataset_test, device, class_names,
                           indices=sample_indices, score_threshold=0.5)
    fig.suptitle(name, y=1.01)
    fig.savefig(OUTPUT_DIR_EVAL / f"{index:02d}_qualitative.png", dpi=160, bbox_inches="tight")
    display(fig)
    plt.close(fig)

### 6.3 Model Comparison

Compare only trained models using the same label space and test split. These are descriptive final test results; model selection belongs on validation data.

In [ ]:
comparison = report["summary"].pivot(index="model", columns="iou_type", values=["mAP", "AP50", "AP75", "AR100"])
display(comparison.round(4))
fig = plot_model_comparison(report["summary"])
display(fig)
plt.close(fig)

## 7. Discussion

### 7.1 Error Analysis and Limitations

The committed train/test split is separated by source video. However, the supplied official validation split overlaps videos from both train and test; the audit reports the current counts (97 training videos and 16 test videos in the current annotations). Thus validation-based tuning is not fully independent of test source videos. Report this limitation with the metrics. A strictly independent experiment requires a new three-way video-level split and retraining; it cannot be fixed by relabelling existing results.

Rare classes and size buckets may have undefined metrics. Empty-annotation images remain in evaluation so false positives still count. Visual examples and class AP identify symptoms; they do not by themselves establish the cause of an error.

In [ ]:
# Highlight classes requiring inspection without treating absent classes as failures.
for (name, iou_type), rows in report["per_class"].groupby(["model", "iou_type"], sort=False):
    print(f"\n{name} / {iou_type}: lowest measured per-class AP")
    display(rows.dropna(subset=["AP"]).nsmallest(5, "AP").round(4))
for message in split_audit["warnings"]:
    print("Split limitation:", message)

## 8. Conclusion and Future Work

## References